In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

protein_path = ROOT / "data/processed/human_proteins_with_string.parquet"
gaf_path = ROOT / "data/raw/go/goa_human.gaf.gz"

protein_df = pd.read_parquet(protein_path)

GAF_COLUMNS = [
    "DB", "DB_Object_ID", "DB_Object_Symbol", "Qualifier",
    "GO_ID", "DB_Reference", "Evidence_Code", "With_or_From",
    "Aspect", "DB_Object_Name", "DB_Object_Synonym",
    "DB_Object_Type", "Taxon", "Date", "Assigned_By",
    "Annotation_Extension", "Gene_Product_Form_ID",
]

gaf_df = pd.read_csv(
    gaf_path,
    sep="\t",
    comment="!",
    header=None,
    names=GAF_COLUMNS,
    compression="gzip",
    dtype=str,
)

In [2]:
protein_ids = set(protein_df["uniprot_id"])

# Positive, matched Molecular Function annotations only
is_not_annotation = gaf_df["Qualifier"].fillna("").str.contains(
    r"(?:^|\|)NOT(?:\||$)",
    regex=True,
)

mf_annotations = gaf_df[
    (~is_not_annotation)
    & (gaf_df["Aspect"] == "F")
    & (gaf_df["DB_Object_ID"].isin(protein_ids))
][
    ["DB_Object_ID", "DB_Object_Symbol", "GO_ID", "Evidence_Code"]
].drop_duplicates(
    subset=["DB_Object_ID", "GO_ID"]
).copy()

mf_annotations = mf_annotations.rename(
    columns={
        "DB_Object_ID": "uniprot_id",
        "DB_Object_Symbol": "gene_name",
        "GO_ID": "go_id",
        "Evidence_Code": "evidence_code",
    }
)

print("Unique protein-GO MF labels:", f"{len(mf_annotations):,}")
print("Proteins with at least one MF label:", f"{mf_annotations['uniprot_id'].nunique():,}")
print("Unique Molecular Function GO terms:", f"{mf_annotations['go_id'].nunique():,}")

Unique protein-GO MF labels: 75,280
Proteins with at least one MF label: 17,560
Unique Molecular Function GO terms: 4,810


In [3]:
# Har GO term ke saath kitne unique proteins associated hain
term_support = (
    mf_annotations.groupby("go_id")["uniprot_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("GO-term support statistics:")
print(term_support.describe())

thresholds = [10, 25, 50, 100, 250, 500, 1000]

threshold_summary = pd.DataFrame(
    {
        "minimum_proteins_per_term": thresholds,
        "GO_terms_retained": [
            (term_support >= threshold).sum()
            for threshold in thresholds
        ],
    }
)

threshold_summary

GO-term support statistics:
count     4810.000000
mean        15.650728
std        206.380749
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max      13643.000000
Name: uniprot_id, dtype: float64


,minimum_proteins_per_term,GO_terms_retained
0,10,891
1,25,370
2,50,174
3,100,98
4,250,45
5,500,15
6,1000,7


In [4]:
# Top 25 most frequent Molecular Function labels
term_support.head(25).rename("protein_count").to_frame()

,protein_count
go_id,
GO:0005515,13643
GO:0042802,1777
GO:0003723,1386
GO:0000981,1197
GO:0003677,1176
GO:0005524,1137
GO:0000978,1085
GO:0008270,841
GO:0004930,759
